# NB00_setup
Shared settings and helper functions. No data is loaded here.
Other notebooks call it with `%run NB00_setup` (it runs inside the caller's session).

In [ ]:
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

# ---------- settings ----------
V2_EFFECTIVE = "2026-11-03 12:00:00"        # effective date used for drivers_v2 (the files carry no real change date)
HIGH_DATE    = "9999-12-31 23:59:59"        # effective_to of the current row of a Type 2 dimension
DATE_START, DATE_END = "2026-01-01", "2026-12-31"   # initial range of dim_date

# ---------- helpers ----------
def table_exists(name):
    return spark.catalog.tableExists(name)

def max_key(table, key_col):
    """Largest surrogate key in a table (0 when the table does not exist or is empty)."""
    if not table_exists(table):
        return 0
    return spark.table(table).agg(F.coalesce(F.max(key_col), F.lit(0))).first()[0]

def merge_simple(df, table, keys):
    """MERGE on a natural key (for static dimensions). Running it twice does not duplicate rows."""
    keys = [keys] if isinstance(keys, str) else keys
    if not table_exists(table):
        df.write.format("delta").saveAsTable(table)
        return
    cols = [c for c in df.columns if c not in keys]
    cond = " AND ".join([f"t.{k} = s.{k}" for k in keys])
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.alias("s"), cond)
        .whenMatchedUpdate(set={c: f"s.{c}" for c in cols})
        .whenNotMatchedInsertAll()
        .execute())

def merge_type1_surrogate(df, table, nat_key, sk):
    """Type 1: overwrite old values, and give a surrogate key only to new rows."""
    cols = df.columns
    if not table_exists(table):
        out = (df.withColumn(sk, F.row_number().over(Window.orderBy(nat_key)).cast("long"))
                 .select(sk, *cols))
        out.write.format("delta").saveAsTable(table)
        return
    existing = spark.table(table).select(nat_key, sk)
    src = df.join(existing, nat_key, "left").select(sk, *cols)
    base = max_key(table, sk)
    old = src.where(F.col(sk).isNotNull())
    new = (src.where(F.col(sk).isNull())
              .withColumn(sk, (F.row_number().over(Window.orderBy(nat_key)) + F.lit(base)).cast("long"))
              .select(sk, *cols))
    (DeltaTable.forName(spark, table).alias("t")
        .merge(old.unionByName(new).alias("s"), f"t.{nat_key} = s.{nat_key}")
        .whenMatchedUpdate(set={c: f"s.{c}" for c in cols if c != nat_key})
        .whenNotMatchedInsertAll()
        .execute())

def show_history(table, n=10):
    """Delta table history (version, time, operation, metrics) - used as time-travel evidence."""
    (spark.sql(f"DESCRIBE HISTORY {table}")
          .select("version", "timestamp", "operation", "operationMetrics")
          .show(n, truncate=False))

print("NB00 ready")